# Notebook 04: Memory Implementation

## Learning Objectives
- Set up AgentCore Memory for user preferences
- Implement conversation context management
- Add personalization based on user history
- Create memory hooks for automatic storage/retrieval
- Integrate memory with travel agent

## Prerequisites
- Completed Notebook 03 (Gateway Integration)
- Gateway with external APIs configured
- Travel agent deployed to AgentCore Runtime
This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


## Step 1: Connect to your AWS environment

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

console.log("\u2705 AWS Profile set. Please restart kernel and run all cells.");

## Step 2: Import libraries

In [ ]:
import { MemoryClient, type MemoryStrategyDict, StrategyType } from "../toolkit/mod.ts";
import { loadEnv, state, writeFile } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

console.log("\u2705 AgentCore Memory imports successful");

## Step 3: Create Travel Memory Resource

In [ ]:
// Configuration
const REGION = "us-east-1";
const MEMORY_NAME = "TravelMateMemory";
const USER_ID = "travel_user_001";
const SESSION_ID = `travel_${new Date().toISOString().replace(/[-:TZ.]/g, "").slice(0, 14)}`;

console.log("\ud83e\udde0 Creating AgentCore Memory for Travel Agent");
console.log(`Memory Name: ${MEMORY_NAME}`);
console.log(`Region: ${REGION}`);
console.log(`User ID: ${USER_ID}`);

// Initialize Memory Client
const client = new MemoryClient({ region: REGION });
console.log("\u2705 Memory client initialized");

In [ ]:
// Define memory strategies for travel planning
const strategies: MemoryStrategyDict[] = [
  {
    [StrategyType.USER_PREFERENCE]: {
      name: "TravelPreferences",
      description: "Captures user travel preferences and behavior",
      namespaces: ["travel/user/{actorId}/preferences"],
    },
  },
  {
    [StrategyType.SEMANTIC]: {
      name: "TravelSemantic",
      description: "Stores travel facts and trip information",
      namespaces: ["travel/user/{actorId}/semantic"],
    },
  },
  {
    [StrategyType.SUMMARY]: {
      name: "TravelSummary",
      description: "Maintains conversation summaries for context",
      namespaces: ["travel/user/{actorId}/summary/{sessionId}"],
    },
  },
];

console.log("\ud83d\udccb Memory Strategies Defined:");
for (const strategy of strategies) {
  const [strategyType, config] = Object.entries(strategy)[0];
  console.log(`  \u2022 ${config.name} (${strategyType})`);
  console.log(`    ${config.description}`);
}

In [ ]:
// Create memory resource
let memoryId: string;
try {
  console.log("\ud83d\ude80 Creating memory resource...");
  const memory = await client.createMemoryAndWait({
    name: MEMORY_NAME,
    strategies,
    description: "Memory for AI Travel Companion agent",
    eventExpiryDays: 365, // Keep travel memories for 1 year
  });
  memoryId = (memory.id ?? memory.memoryId)!;
  console.log(`\u2705 Created memory: ${memoryId}`);
} catch (error) {
  // If memory already exists, retrieve its ID
  if (error instanceof Error && error.message.includes("already exists")) {
    const memories = await client.listMemories();
    const existing = memories.find((m) => (m.id ?? m.memoryId ?? "").startsWith(MEMORY_NAME));
    memoryId = (existing?.id ?? existing?.memoryId)!;
    console.log(`Memory already exists. Using existing memory ID: ${memoryId}`);
  } else {
    throw error;
  }
}

## Step 4: Verify Memory Strategies

In [ ]:
// Verify memory strategies are configured correctly
const strategiesInfo = await client.getMemoryStrategies(memoryId);

console.log("\ud83d\udd0d Memory Strategies Verification:");
console.log("=".repeat(50));
for (const strategy of strategiesInfo) {
  console.log(`\n\ud83d\udccc ${strategy.name}`);
  console.log(`   Type: ${strategy.type ?? strategy.memoryStrategyType}`);
  console.log(`   Description: ${strategy.description}`);
  console.log(`   Namespaces: ${JSON.stringify(strategy.namespaces ?? strategy.namespaceTemplates)}`);
}

console.log(`\n\u2705 ${strategiesInfo.length} strategies configured successfully`);

## Step 5: Seed Travel Preferences

In [ ]:
// Seed initial travel preferences for demonstration.
// Messages keep Python's (text, role) shape, so this cell reads the same.
import { travelInteractions } from "../backend/memory/memory_setup.ts";

console.log(`\ud83c\udf31 Seeding ${travelInteractions.length} travel interactions...`);

await client.createEvent({
  memoryId,
  actorId: USER_ID,
  sessionId: "preference_setup",
  messages: travelInteractions,
});

console.log(`\u2705 Seeded travel preferences for user: ${USER_ID}`);

## Step 6: Test Memory Retrieval

In [ ]:
// Test memory retrieval
console.log("\ud83e\uddea Testing Memory Retrieval");
console.log("=".repeat(50));

// Wait for memory processing: extraction runs asynchronously after the event is stored
await new Promise((resolve) => setTimeout(resolve, 30_000));

// Try to retrieve memories
try {
  const memories = await client.retrieveMemories({
    memoryId,
    namespace: `travel/user/${USER_ID}/preferences`,
    query: "vegetarian food preferences",
    topK: 3,
  });

  console.log(`\n\ud83d\udcda Retrieved ${memories.length} memories:`);
  memories.forEach((memory, index) => {
    const text = (memory.content as { text?: string } | undefined)?.text ?? "";
    console.log(`  ${index + 1}. ${text.slice(0, 100)}...`);
  });
} catch (error) {
  console.log(`\u26a0\ufe0f Memory retrieval test: ${error}`);
  console.log("This is normal - memories may take time to process");
}

## Step 7: Save Memory Information

In [ ]:
// Save memory information for use in subsequent notebooks
const memoryInfo = {
  memory_id: memoryId,
  memory_name: MEMORY_NAME,
  region: REGION,
  user_id: USER_ID,
  session_id: SESSION_ID,
};

// Save to file for next notebooks
await writeFile("environments/memory_info.json", `${JSON.stringify(memoryInfo, null, 2)}\n`);
await state.set("memory_info", memoryInfo);

console.log("\ud83d\udcbe Memory information saved to environments/memory_info.json");
console.log("\n\ud83d\udccb Memory Summary:");
console.log(`  Memory ID: ${memoryInfo.memory_id}`);
console.log(`  User ID: ${memoryInfo.user_id}`);
console.log(`  Region: ${memoryInfo.region}`);

## Summary

✅ **Completed in this notebook:**
- AgentCore Memory resource with 3 strategies
- User preference storage and seeding
- Memory verification and basic testing
- Memory information saved for next notebooks

➡️ **Next: Notebook 05 - Identity & OAuth**
- Set up Google Drive OAuth integration
- Implement secure credential management
- Add itinerary storage to Google Drive
- Integrate identity with travel agent